# 价值因子研究

**因子体系**：EP（盈利收益率）+ BP（净资产收益率）+ SP（销售收益率）合成价值因子  
**核心逻辑**：低估值股票（高 EP/BP/SP）相对市场具有长期超额收益  
**作者**：jialong / xingyu  
**日期**：2026-03

---
## Section 0：参数设置

In [ ]:
# ── 全局参数 ─────────────────────────────────────────────────
SYMBOLS = [
    "000001", "000002", "000063", "000333", "000651",
    "000858", "001979", "002415", "002594", "600000",
    "600009", "600016", "600028", "600030", "600036",
    "600048", "600050", "600104", "600276", "600309",
]

START = "2020-01-01"
END   = "2024-12-31"

# 合成权重（等权）
VALUE_WEIGHTS = (1/3, 1/3, 1/3)

# IC 分析的前瞻收益天数
FORWARD_DAYS = 20  # 约一个月

print(f"股票池大小: {len(SYMBOLS)}")
print(f"回测区间: {START} ~ {END}")

---
## Section 1：数据加载 — PE / PB / PS 宽表

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── 用 mock 数据演示（需要真实数据时替换为下方注释块）──────────────
# 需要真实数据
np.random.seed(2026)
dates = pd.bdate_range(START, END)
n_dates, n_syms = len(dates), len(SYMBOLS)

# 模拟 PE：主要分布在 10~50，偶有负值（亏损股）
pe_raw = pd.DataFrame(
    np.where(
        np.random.rand(n_dates, n_syms) < 0.05,   # 5% 概率为负（亏损股）
        -np.abs(np.random.randn(n_dates, n_syms)) * 10,
        np.abs(np.random.randn(n_dates, n_syms)) * 15 + 15,
    ),
    index=dates, columns=SYMBOLS,
)

# 模拟 PB：主要分布在 0.5~5，偶有负值（资不抵债）
pb_raw = pd.DataFrame(
    np.where(
        np.random.rand(n_dates, n_syms) < 0.03,
        -np.abs(np.random.randn(n_dates, n_syms)),
        np.abs(np.random.randn(n_dates, n_syms)) * 1.5 + 1.5,
    ),
    index=dates, columns=SYMBOLS,
)

# 模拟 PS：主要分布在 0.5~8
ps_raw = pd.DataFrame(
    np.abs(np.random.randn(n_dates, n_syms)) * 2 + 2,
    index=dates, columns=SYMBOLS,
)

print(f"PE 宽表形状: {pe_raw.shape}")
print(f"PB 宽表形状: {pb_raw.shape}")
print(f"PS 宽表形状: {ps_raw.shape}")

In [ ]:
# ── 真实数据加载（需要配置好 fundamental_loader 缓存后取消注释）─────
# 需要真实数据
#
# import sys; sys.path.insert(0, '../../..')
# from utils.fundamental_loader import get_pe_pb
#
# pe_list, pb_list = [], []
# for sym in SYMBOLS:
#     df = get_pe_pb(sym, start=START, end=END)
#     if df is not None and len(df) > 0:
#         pe_list.append(df['pe_ttm'].rename(sym))
#         pb_list.append(df['pb'].rename(sym))
#
# pe_raw = pd.concat(pe_list, axis=1).reindex(dates).ffill()
# pb_raw = pd.concat(pb_list, axis=1).reindex(dates).ffill()
# # PS = pcf 字段（市现率）或另行计算市销率
# ps_raw = ...  # 需要真实 PS 数据
#
# # 数据质量门
# assert pe_raw.shape[0] > 100
# assert pe_raw.isnull().mean().max() < 0.3
# print(f'✅ 真实数据 OK | 行数: {pe_raw.shape[0]} | 时间: {pe_raw.index[0].date()} ~ {pe_raw.index[-1].date()}')

print("使用 mock 数据（已标注 # 需要真实数据 的 cell）")

---
## Section 2：计算 EP / BP / SP 及合成价值因子

In [ ]:
import sys
sys.path.insert(0, '../../..')
from research.factors.value.value_factor import (
    compute_ep, compute_bp, compute_sp, compute_composite_value
)

ep = compute_ep(pe_raw)
bp = compute_bp(pb_raw)
sp = compute_sp(ps_raw)
value = compute_composite_value(ep, bp, sp, weights=VALUE_WEIGHTS)

print(f"EP  — 非空比例: {ep.notna().mean().mean():.1%}")
print(f"BP  — 非空比例: {bp.notna().mean().mean():.1%}")
print(f"SP  — 非空比例: {sp.notna().mean().mean():.1%}")
print(f"价值合成因子 — 形状: {value.shape}, 非空比例: {value.notna().mean().mean():.1%}")

In [ ]:
import matplotlib.pyplot as plt

# 取某一截面日查看分布
snapshot_date = value.dropna(how='all').index[len(value.dropna(how='all')) // 2]
cross_section = value.loc[snapshot_date].dropna()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, fac) in zip(axes, [("EP", ep), ("BP", bp), ("合成价值因子", value)]):
    cs = fac.loc[snapshot_date].dropna()
    ax.hist(cs, bins=15, edgecolor='black', alpha=0.7)
    ax.set_title(f"{name} 截面分布 ({snapshot_date.date()})")
    ax.set_xlabel("因子值")
    ax.set_ylabel("频数")
    ax.axvline(cs.mean(), color='red', linestyle='--', label=f'均值={cs.mean():.3f}')
    ax.legend()

plt.tight_layout()
plt.show()
print(f"截面日: {snapshot_date.date()} | 有效股票数: {len(cross_section)}")

---
## Section 3：IC / ICIR 分析

In [ ]:
from utils.factor_analysis import compute_ic_series, ic_summary

# 构造前瞻收益宽表（需要真实价格数据时替换）
# 需要真实数据
np.random.seed(99)
# mock 收益率：价值因子高 → 未来收益略高（模拟正相关）
ret_mock = pd.DataFrame(
    np.random.randn(n_dates, n_syms) * 0.02,
    index=dates, columns=SYMBOLS,
)
# 给价值因子高的股票加一点正向漂移，模拟正 IC
value_rank = value.rank(axis=1, pct=True)
signal_shift = value_rank.shift(1) * 0.005
ret_mock = ret_mock + signal_shift.fillna(0)

# 计算 FORWARD_DAYS 期前瞻收益
fwd_ret = ret_mock.rolling(FORWARD_DAYS).sum().shift(-FORWARD_DAYS)

ic_series = compute_ic_series(value, fwd_ret, method='spearman')

summary = ic_summary(ic_series)
print("IC 统计摘要:")
for k, v in summary.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# IC 时序
ic_series.dropna().plot(ax=axes[0], alpha=0.6)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].axhline(ic_series.mean(), color='red', linestyle='--',
                label=f'均值={ic_series.mean():.4f}')
axes[0].set_title(f"价值因子 Rank IC 时序 (前瞻{FORWARD_DAYS}日)")
axes[0].legend()

# IC 累积
ic_series.dropna().cumsum().plot(ax=axes[1], color='darkorange')
axes[1].set_title("Rank IC 累积")

plt.tight_layout()
plt.show()

icir = ic_series.mean() / ic_series.std()
print(f"IC 均值: {ic_series.mean():.4f}")
print(f"ICIR: {icir:.4f}")

---
## Section 4：分层回测（10 组）

In [ ]:
from utils.factor_analysis import quintile_backtest

# 构造每日收益率（shift(1) 避免未来函数）
daily_ret = ret_mock.copy()

# 分层回测（10 组）
layer_ret = quintile_backtest(value, daily_ret, n_groups=10)

print(f"分层收益率形状: {layer_ret.shape}")
print("各分组年化收益率（mock 数据仅供验证流程）:")
ann_ret = (layer_ret + 1).prod() ** (252 / len(layer_ret)) - 1
print(ann_ret.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 各分组累积净值
(1 + layer_ret).cumprod().plot(ax=axes[0])
axes[0].set_title("价值因子 10 组分层累积净值")
axes[0].set_ylabel("净值")
axes[0].legend(loc='upper left', fontsize=8)

# 多空组合（第10组 - 第1组）
ls_ret = layer_ret.iloc[:, -1] - layer_ret.iloc[:, 0]
(1 + ls_ret).cumprod().plot(ax=axes[1], color='purple')
axes[1].set_title("多空组合净值（Top - Bottom）")
axes[1].axhline(1, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

ls_ann = (1 + ls_ret).prod() ** (252 / len(ls_ret)) - 1
ls_sharpe = ls_ret.mean() / ls_ret.std() * np.sqrt(252)
print(f"多空年化收益: {ls_ann:.2%}")
print(f"多空夏普比率: {ls_sharpe:.2f}")

---
## Section 5：与动量因子的相关性（正交性检验）

In [ ]:
# 需要真实数据（此处用 mock 价格生成动量因子演示）
from research.factors.momentum.momentum_factor import compute_momentum

# 模拟价格序列
np.random.seed(7)
prices_mock = pd.DataFrame(
    (100 + np.random.randn(n_dates, n_syms).cumsum(axis=0)),
    index=dates, columns=SYMBOLS,
)

# 计算 20 日动量因子
mom_20 = compute_momentum(prices_mock, lookback=20, skip=1)

# 截面相关性：每日计算价值因子与动量因子的截面 Spearman 相关
from scipy.stats import spearmanr

common_dates = value.index.intersection(mom_20.index)
daily_corr = []
for d in common_dates:
    v_cross = value.loc[d].dropna()
    m_cross = mom_20.loc[d].dropna()
    common_sym = v_cross.index.intersection(m_cross.index)
    if len(common_sym) < 10:
        daily_corr.append(np.nan)
        continue
    corr, _ = spearmanr(v_cross[common_sym], m_cross[common_sym])
    daily_corr.append(corr)

corr_series = pd.Series(daily_corr, index=common_dates)
print(f"价值 vs 动量 平均截面相关性: {corr_series.mean():.4f}")
print(f"（接近 0 表明两因子正交，可互补合成）")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
corr_series.dropna().rolling(20).mean().plot(ax=ax, label='20日滚动均值')
corr_series.dropna().plot(ax=ax, alpha=0.3, label='日度相关')
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(corr_series.mean(), color='red', linestyle='--',
           label=f'总体均值={corr_series.mean():.4f}')
ax.set_title("价值因子 vs 动量因子 截面 Spearman 相关性")
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 6：结论

### 因子构建

| 子因子 | 公式 | 覆盖率（mock） |
|--------|------|---------------|
| EP     | 1 / PE_TTM（PE≤0置NaN） | ~95% |
| BP     | 1 / PB（PB≤0置NaN）     | ~97% |
| SP     | 1 / PS                  | ~100% |
| 合成   | 截面z-score等权合成      | — |

### 预期表现（待真实数据回测后填入）

| 指标 | 预期区间 | 实测值 |
|------|---------|-------|
| IC 均值 | 0.02 ~ 0.06 | [待填入实际回测结果] |
| ICIR | 0.3 ~ 0.8 | [待填入实际回测结果] |
| 多空年化 | 10% ~ 30% | [待填入实际回测结果] |
| 多空夏普 | 0.5 ~ 1.5 | [待填入实际回测结果] |

### 与动量因子正交性

- 价值因子（低估值）与动量因子（近期涨幅）在理论上负相关
- 实测截面相关系数预期在 -0.1 ~ 0.1 之间，可与动量互补
- [待填入实际回测结果]

### 局限性

1. **PS 数据质量**：`fundamental_loader` 当前返回 `pcf`（市现率）而非 `ps`（市销率），需额外接入市销率数据源
2. **财务数据频率**：PE/PB 为日度估值，但背后财务数据为季度更新，存在信息滞后
3. **行业偏差**：价值因子易暴露于金融、地产等高杠杆行业，需行业中性化后使用
4. **股票池**：当前使用抽样股票池，全 A 股结果可能差异较大
5. **样本量**：mock 数据仅 20 只股票，实际需要 200+ 只才能得到稳健的 IC 估计